In [1]:
import pandas as pd
import numpy as np
import arviz as az
from jax import numpy as jnp
from jax.random import PRNGKey
import numpyro
from numpyro import distributions as dist
from numpyro import infer

pd.options.plotting.backend = "plotly"

from summer3.graph import Parameter, defer, CompartmentValues
from summer3.epi import CompartmentalModelODE, CategoryData, \
    strat_data_from_pandas, build_istate, dti_to_epoch, TransitionFlow, \
    CategoryGroup, StratSpec, ManagedArray

from tb_macro.constants import ALL_COMPARTMENTS, AGE_STRATA, INFECT_COMPS, \
    DATA_PATH
from tb_macro.epi import get_base_model, \
    add_natural_history, add_health_system_flows, add_ageing_flows, \
    add_seeding, add_detection
from tb_macro.inputs import get_country_pop, get_single_age_pop_from_ungroups, \
    get_group_popsizes, get_un_mortality, load_conmat, convert_conmat, \
    normalise_spectral_radius, add_groups_to_single_pop, build_age_weight_lookup
from tb_macro.mixing import get_norm_c_matrix

In [2]:
iso3 = "VNM"
start_year = 1950

pop_data = get_country_pop(iso3)
single_age = get_single_age_pop_from_ungroups(pop_data)
group_popsize = get_group_popsizes(single_age)
agg_pop_data = group_popsize.sum(axis=1)
mort_data = get_un_mortality(iso3, start_year)
death_rates = mort_data.div(group_popsize, axis=0).dropna()
conmat_data = load_conmat("KIR") # temporarily using Kiribati
matrix = convert_conmat(conmat_data)
norm_matrix = normalise_spectral_radius(matrix)
add_groups_to_single_pop(single_age)
age_weights = build_age_weight_lookup(single_age)

In [3]:
def get_fertility_data():
    raw_data = pd.read_csv(DATA_PATH / "population/un_fertility_rates_KIR.csv", index_col=0)
    data = raw_data.div(raw_data.sum(axis=1), axis=0)
    data.columns = data.columns.astype(int)
    return data

fert = get_fertility_data()

In [4]:
from tb_macro.constants import MAX_AGE
fert_padded = fert.reindex(columns=range(MAX_AGE + 1), fill_value=0.0)

In [5]:
def infect_process(
    compartment_values: ManagedArray,
    contact_rate: float,
    freq_dens_exponent: float,
    age_cats: CategoryGroup,
    infectious_compartments: StratSpec,
    mm_function: callable,
    a_spread: float,
    bg_mixing: float,
    weights: pd.DataFrame,
    pops: pd.DataFrame,
    fert: pd.DataFrame,
):
    infector_cats = age_cats
    infectee_cats = age_cats
    infect_pop_cats = infector_cats.product(infectious_compartments)
    ipops = compartment_values.sumcats(infect_pop_cats)
    total_pop = compartment_values.sumcats(infector_cats)
    inf_pressure = ipops.data / total_pop.data ** freq_dens_exponent
    time = 2000
    age_foi = mm_function(weights, pops, fert, time, a_spread, bg_mixing) @ inf_pressure * contact_rate
    return CategoryData(infectee_cats, age_foi)

In [ ]:
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model()
for comp in INFECT_COMPS:
    reinfect_foi = defer(infect_process)(
        CompartmentValues, 
        Parameter("contact_rate", 0.0) * Parameter(f"rel_sus_{comp}", 0.0), 
        Parameter("freq_dens_exponent", 1.0), 
        age_strat.categories(), 
        disease_state["active"],
        get_norm_c_matrix,
        Parameter("a_spread", 0.0),
        Parameter("bg_mixing", 0.0),
        age_weights,
        group_popsize,
        fert_padded,
    )
    reinfect = TransitionFlow(
        f"infect_{comp}",
        disease_state[comp],
        disease_state["incipient"],
        reinfect_foi,
    )
    epi_model.add_flow(reinfect)

add_natural_history(epi_model, disease_state, clin_strat, infect_strat)
add_health_system_flows(epi_model, disease_state, clin_strat, infect_strat)
add_ageing_flows(epi_model, age_strat)
add_seeding(epi_model, disease_state)
add_detection(epi_model, disease_state, clin_strat)

In [7]:
age_strings = [str(a) for a in AGE_STRATA]
data = pd.Series(index=age_strings, data=np.array([1000.0] * len(AGE_STRATA)))
base_pops = strat_data_from_pandas(data, age_strat)
init_splits = [0.0] * len(ALL_COMPARTMENTS)
init_splits[ALL_COMPARTMENTS.index("mtb_naive")] = 1.0
pop_splits = [CategoryData(disease_state.categories(), jnp.array((init_splits)))]
epi_model.set_initial_population(base_pops, pop_splits)

In [8]:
def get_runner(epi_model):
    istate = build_istate(epi_model.cmap, epi_model.base_pops, epi_model.pop_splits)
    cmodel = CompartmentalModelODE(epi_model.cmap, epi_model.flows)
    runner = cmodel.get_runner(
        len(epi_model.times), dti_to_epoch(epi_model.times), True
    )
    return runner, istate

In [ ]:
base_params = {
    "contact_rate": 1.0,
    "freq_dens_exponent": 1.0,
    "rel_sus_mtb_naive": 1.0,
    "rel_sus_contained": 0.0, 
    "rel_sus_cleared": 0.0, 
    "rel_sus_recovered": 0.0, 
    "contain": 0.0,
    "clearance_rate": 0.0,
    "breakdown_rate": 0.0,
    "progression": 1.0,
    "increase_infect": 1.0,
    "decrease_infect": 1.0,
    "clinical_development": 1.0,
    "clinical_regression": 1.0,
    "self_recovery": 0.2,
    "detection": 0.0,
    "treatment_recovery": 1.0,
    "treatment_relapse": 0.0,
    "seed_peak_time": 30.0,
    "seed_peak_rate": 0.01,
    "seed_duration": 10.0,
    "recent_detection_rate": 0.7,
    "passive_detection_shape": 0.15,
    "passive_detection_inflection": 55.0,
    "passive_detection_past_frac": 0.5,
    "a_spread": 0.1,
    "bg_mixing": 0.2,
}
runner, istate = get_runner(epi_model)
results = epi_model.run(base_params)

Scatter
Gather
Gather
Gather
Scatter
Gather
Gather
Gather


GraphRunError: ('_var10', AttributeError("'jaxlib._jax.ArrayImpl' object has no attribute 'index'"))

In [ ]:
results["compartments"].sumcats(compartment=disease_state.categories()).to_pandas_df().plot()

In [ ]:
inf_target = (
    results["flows"]["infect_mtb_naive"]
    .sum(to_dims="time")
    .to_pandas_df()
    .rolling(7)
    .sum()[7:60:7]
)["data"]
inf_target_fuzzy = inf_target * np.exp(
    np.random.normal(scale=0.01, size=len(inf_target))
)
inf_target_fuzzy.plot()

In [ ]:
def get_derived_results(params):
    results = epi_model.run(params)  # runner.run(istate.data, params)
    inf_flow = results["flows"]["infect_mtb_naive"]
    weekly_target = (
        inf_flow.sum(to_dims="time").rolling(7, jnp.sum).query(time=inf_target.index)
    )
    return weekly_target

In [ ]:
priors = {
    "contact_rate": dist.Uniform(0.001, 0.5),
}

In [ ]:
def model():
    params = base_params | {k: numpyro.sample(k, v) for k, v in priors.items()}
    weekly_modelled = get_derived_results(params)

    ll = dist.Poisson(inf_target_fuzzy.to_numpy()).log_prob(weekly_modelled.data)
    numpyro.factor("ll", ll)

In [ ]:
kernel = infer.NUTS(model)
mcmc = infer.MCMC(kernel, num_warmup=200, num_samples=200, num_chains=4)
k = PRNGKey(0)
mcmc.run(k)

Scatter
Gather
Gather
Gather


Exception ignored in: <function _xla_gc_callback at 0x30b5de020>
Traceback (most recent call last):
  File "/Users/jamestrauer/dev/tb_macroeconomics/.pixi/envs/default/lib/python3.13/site-packages/jax/_src/lib/__init__.py", line 124, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


Scatter
Gather
Gather
Gather


KeyboardInterrupt: 

In [ ]:
idata = az.from_numpyro(mcmc)

In [ ]:
az.summary(idata)

In [ ]:
az.plot_posterior(idata)

In [ ]:
az.plot_trace(idata, compact=False)